# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [9]:
#%load_ext dotenv
#%dotenv ../05_src/.secrets

import os
from dotenv import load_dotenv

load_dotenv("../05_src/.secrets", override=True)

print(repr(os.getenv("OPENAI_API_KEY")))

'any_value'


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [10]:
from langchain_community.document_loaders import PyPDFLoader

#file_path = "../docs/Managing_Oneself_Drucker_HBR.pdf" # not from local file, but from URL.

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(url)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

#print(document_text) #I am not printing the whole document text here, but you can see it in your environment.


13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [11]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional
import os

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# ── Output schema ──────────────────────────────────────────────────────────────
class ArticleSummary(BaseModel):
    Author: str = Field(description="The author(s) of the article")
    Title: str = Field(description="The title of the article")
    Relevance: str = Field(description="A paragraph explaining why this article is relevant for an AI professional's development")
    Summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens, written in the requested tone")
    Tone: str = Field(description="The tone used to write the summary")
    InputTokens: Optional[int] = Field(default=None, description="Number of input tokens (populated from response usage)")
    OutputTokens: Optional[int] = Field(default=None, description="Number of output tokens (populated from response usage)")

# ── Prompts ────────────────────────────────────────────────────────────────────
tone = "Legalese"

system_prompt = (
    "You are an expert analyst for AI professionals. "
    "Extract structured information from the provided article exactly as instructed."
)

user_prompt = f"""
Given the article text below, produce a structured response with the following fields:

- Author: the author(s) of the article.
- Title: the title of the article.
- Relevance: a single paragraph explaining why this article is relevant for an AI professional in their professional development.
- Summary: a concise summary no longer than 1000 tokens, written entirely in the following tone: {tone}.
- Tone: write exactly "{tone}".

Article:
<article>
{document_text}
</article>
"""

# ── API call using structured output (.parse) ──────────────────────────────────
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=system_prompt,
    input=user_prompt,
    text_format=ArticleSummary,
)

# Retrieve the Pydantic object and populate token counts from response usage
article_summary = response.output_parsed
article_summary = article_summary.model_copy(update={
    "InputTokens": response.usage.input_tokens,
    "OutputTokens": response.usage.output_tokens,
})

article_summary


ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is critically relevant for AI professionals as it emphasizes the importance of self-knowledge, understanding strengths, and aligning personal values with professional goals. In the rapidly evolving field of AI, the ability to self-manage and adapt one's skills is essential for sustained success and leadership.", Summary="In this discourse, it is posited that individuals in the contemporary workforce must assume the mantle of their own chief executive officer, necessitating a thorough comprehension of personal strengths, performance modalities, and inherent values. The article delineates a systematic methodology—termed feedback analysis—whereby one must document anticipations of outcomes associated with key decisions and subsequently juxtapose these against actual results. This analytical exercise serves as a catalyst for uncovering true strengths and identifying non-productive habits or deficien

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [12]:
# Extract fields directly from the Pydantic object — no regex needed
title = article_summary.Title
author = article_summary.Author
relevance = article_summary.Relevance
summary_text = article_summary.Summary
tone = article_summary.Tone

print("TITLE:", title)
print("AUTHOR:", author)
print("RELEVANCE:", relevance)
print("SUMMARY:", summary_text)
print("TONE:", tone)
print("INPUT TOKENS:", article_summary.InputTokens)
print("OUTPUT TOKENS:", article_summary.OutputTokens)


TITLE: Managing Oneself
AUTHOR: Peter F. Drucker
RELEVANCE: This article is critically relevant for AI professionals as it emphasizes the importance of self-knowledge, understanding strengths, and aligning personal values with professional goals. In the rapidly evolving field of AI, the ability to self-manage and adapt one's skills is essential for sustained success and leadership.
SUMMARY: In this discourse, it is posited that individuals in the contemporary workforce must assume the mantle of their own chief executive officer, necessitating a thorough comprehension of personal strengths, performance modalities, and inherent values. The article delineates a systematic methodology—termed feedback analysis—whereby one must document anticipations of outcomes associated with key decisions and subsequently juxtapose these against actual results. This analytical exercise serves as a catalyst for uncovering true strengths and identifying non-productive habits or deficiencies. Furthermore, th

In [13]:
from deepeval.models import GPTModel

eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


In [14]:
#Deepeval libs
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [15]:
#Creating a test case for the summary evaluation using the extracted summary_text as the actual output and the original document_text as the input.
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text
)


In [16]:
#Summarization metric with 5 bespoke questions that are relevant for evaluating the quality of a summary, along with a threshold of 0.5 to determine if the summary is acceptable or not based on the average score across these questions.
summarization_questions = [
    "Does the summary accurately capture the central idea of the article?",
    "Does the summary include the most important supporting arguments and ideas?",
    "Does the summary avoid introducing claims that are not supported by the article?",
    "Does the summary preserve the meaning and intent of the original article?",
    "Is the summary concise while still covering the essential points?"
]

summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    threshold=0.5,
    model=eval_model,
)

summarization_metric.measure(test_case)

print("Summarization score:", summarization_metric.score)
print("Summarization reason:", summarization_metric.reason)


Output()

Summarization score: 1.0
Summarization reason: The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
from pydantic import BaseModel

class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(document_text, summary_text, tone, eval_model):
    from deepeval.metrics import SummarizationMetric, GEval
    from deepeval.test_case import LLMTestCase, LLMTestCaseParams

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text
    )

    summarization_questions = [
        "Does the summary accurately capture Drucker's main argument about self-management?",
        "Does the summary preserve the article's key ideas about strengths, performance, values, and contribution?",
        "Does the summary avoid adding claims not explicitly supported by the article?",
        "Does the summary preserve the intended meaning of the original article?",
        "Is the summary concise while still covering the essential points?"
    ]

    summarization_metric = SummarizationMetric(
        assessment_questions=summarization_questions,
        threshold=0.5,
        model=eval_model,
    )
    summarization_metric.measure(test_case)

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=[
            "Check whether the summary follows a logical order.",
            "Check whether the ideas connect clearly from sentence to sentence.",
            "Check whether the writing is easy to understand.",
            "Check whether the summary avoids contradictions or confusing statements.",
            "Check whether the overall structure is clear and organized."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    coherence_metric.measure(test_case)

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=[
            f"Check whether the summary reflects the requested tone: {tone}.",
            "Check whether the tone is consistent throughout the summary.",
            "Check whether the tone is clearly distinguishable.",
            "Check whether the tone supports readability.",
            "Check whether the style matches the requested writing approach."
        ],
        evaluation_params=[
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    tonality_metric.measure(test_case)

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids harmful or offensive language.",
            "Check whether the summary avoids fabricated claims.",
            "Check whether the summary does not misrepresent the article in a misleading way.",
            "Check whether the summary is appropriate for an academic setting.",
            "Check whether the summary avoids unsafe or inappropriate content."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ],
        model=eval_model,
    )
    safety_metric.measure(test_case)

    return EvaluationOutput(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason
    )

In [18]:
initial_evaluation = evaluate_summary(
    document_text=document_text,
    summary_text=summary_text,
    tone=tone,
    eval_model=eval_model
)

print(initial_evaluation.model_dump())

Output()

Output()

Output()

Output()

{'SummarizationScore': 1.0, 'SummarizationReason': 'The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment.', 'CoherenceScore': 0.8546255298692722, 'CoherenceReason': 'The response follows a logical order and connects ideas clearly, effectively summarizing the key points of the article. It is easy to understand and avoids contradictions, maintaining clarity throughout. The structure is organized, addressing the main themes of self-management, feedback analysis, and the importance of aligning personal values with organizational values. However, it could benefit from slightly more detail on specific examples or methodologies mentioned in the original text to enhance comprehension.', 'TonalityScore': 0.8189373388327443, 'TonalityReason': 'The summary effectively employs a legalese tone, characterized by formal language and complex sentence structures, which aligns well with the requeste

In [19]:
#Bulding an enhancement prompt to improve the summary based on the evaluation results, asking the model to specifically address the weaknesses identified in the evaluation while maintaining the requested tone and ensuring that the summary is concise and accurate.

enhancement_system_prompt = """
You are a careful editor.
Revise summaries using evaluation feedback.
Your goal is to improve factual faithfulness, coherence, tone consistency, and safety.
Do not add new information not supported by the source article.
"""

enhancement_user_prompt = f"""
You will improve a summary of an article using the original article and evaluation feedback.

Requested tone:
{tone}

Original article:
<article>
{document_text}
</article>

Current summary:
<current_summary>
{summary_text}
</current_summary>

Evaluation feedback:
<summarization_score>{initial_evaluation.SummarizationScore}</summarization_score>
<summarization_reason>{initial_evaluation.SummarizationReason}</summarization_reason>

<coherence_score>{initial_evaluation.CoherenceScore}</coherence_score>
<coherence_reason>{initial_evaluation.CoherenceReason}</coherence_reason>

<tonality_score>{initial_evaluation.TonalityScore}</tonality_score>
<tonality_reason>{initial_evaluation.TonalityReason}</tonality_reason>

<safety_score>{initial_evaluation.SafetyScore}</safety_score>
<safety_reason>{initial_evaluation.SafetyReason}</safety_reason>

Instructions:
1. Rewrite the summary so it is more faithful to the article.
2. Fix any issues mentioned in the evaluation reasons.
3. Keep the summary concise.
4. Do not include title, author, or relevance.
5. Return only the improved summary text.
"""


In [20]:
#Enhanced summary

enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=enhancement_system_prompt,
    input=enhancement_user_prompt,
    temperature=0
)

enhanced_summary_text = enhanced_response.output_text.strip()

print(enhanced_summary_text)

In the contemporary workforce, individuals are urged to adopt the role of their own chief executive officer, necessitating a comprehensive understanding of personal strengths, performance modalities, and intrinsic values. The article delineates a systematic methodology known as feedback analysis, wherein individuals document their anticipated outcomes from key decisions and subsequently compare these expectations with actual results. This analytical exercise serves to reveal true strengths while identifying unproductive habits or deficiencies. Furthermore, the text emphasizes the distinct modalities through which individuals operate and learn, asserting that a profound understanding of one's work style is imperative for optimal performance. The examination of personal ethics and values highlights the necessity for alignment between an individual’s values and those of the employing organization to mitigate frustration and enhance overall job performance. Ultimately, the discourse advoca

In [21]:
#Evaluating the new summary with the same function
enhanced_evaluation = evaluate_summary(
    document_text=document_text,
    summary_text=enhanced_summary_text,
    tone=tone,
    eval_model=eval_model
)

print(enhanced_evaluation.model_dump())


Output()

Output()

Output()

Output()

{'SummarizationScore': 1.0, 'SummarizationReason': 'The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment.', 'CoherenceScore': 0.8790708001161874, 'CoherenceReason': 'The response follows a logical order and connects ideas clearly, effectively summarizing the key concepts of self-management and feedback analysis from the article. The writing is easy to understand and avoids contradictions, maintaining clarity throughout. The overall structure is organized, addressing the main points of strengths, performance modalities, and values in a coherent manner. However, a slight improvement could be made in explicitly linking the conclusion back to the importance of self-management in the knowledge economy, which would enhance the overall cohesiveness.', 'TonalityScore': 0.3439711657447736, 'TonalityReason': 'The summary lacks the formal and precise tone characteristic of legalese, which is 

In [22]:
#Comparison

comparison = {
    "OriginalSummary": summary_text,
    "EnhancedSummary": enhanced_summary_text,
    "InitialEvaluation": initial_evaluation.model_dump(),
    "EnhancedEvaluation": enhanced_evaluation.model_dump(),
}

comparison

{'OriginalSummary': "In this discourse, it is posited that individuals in the contemporary workforce must assume the mantle of their own chief executive officer, necessitating a thorough comprehension of personal strengths, performance modalities, and inherent values. The article delineates a systematic methodology—termed feedback analysis—whereby one must document anticipations of outcomes associated with key decisions and subsequently juxtapose these against actual results. This analytical exercise serves as a catalyst for uncovering true strengths and identifying non-productive habits or deficiencies. Furthermore, the text elucidates the distinct modalities through which individuals operate and learn, asserting that a profound understanding of one's work style is imperative for optimal performance. The examination of personal ethics and values underscores the compatibility requisite between an individual’s values and those of the employing organization in order to avert frustration 

In [23]:
#Cleaner score table
print("Initial Summarization Score:", initial_evaluation.SummarizationScore)
print("Enhanced Summarization Score:", enhanced_evaluation.SummarizationScore)
print()

print("Initial Coherence Score:", initial_evaluation.CoherenceScore)
print("Enhanced Coherence Score:", enhanced_evaluation.CoherenceScore)
print()

print("Initial Tonality Score:", initial_evaluation.TonalityScore)
print("Enhanced Tonality Score:", enhanced_evaluation.TonalityScore)
print()

print("Initial Safety Score:", initial_evaluation.SafetyScore)
print("Enhanced Safety Score:", enhanced_evaluation.SafetyScore)

Initial Summarization Score: 1.0
Enhanced Summarization Score: 1.0

Initial Coherence Score: 0.8546255298692722
Enhanced Coherence Score: 0.8790708001161874

Initial Tonality Score: 0.8189373388327443
Enhanced Tonality Score: 0.3439711657447736

Initial Safety Score: 0.9070269284275472
Enhanced Safety Score: 0.927761947728969


Please, do not forget to add your comments.

### Enhancement results (My comments - E. Fantinatti)

To improve the original summary, I created a second prompt that used three inputs:
1. the original article text,
2. the first generated summary,
3. the evaluation scores and reasons from DeepEval.

The goal was to create a self-correction loop in which the model revised the summary based on evaluator feedback. I then re-ran the same evaluation function on the revised summary and compared the results.

In my case, the enhanced summary [improved / did not improve] the evaluation scores. This happened because the second prompt explicitly instructed the model to correct issues identified by the evaluator, especially around factual faithfulness and clarity.

These controls help, but they are not sufficient on their own. The evaluator is still another model-based judge, so it can be imperfect and somewhat variable. The quality of the feedback also depends on the prompt, the selected metrics, and the source text. For that reason, automated evaluation is useful for iterative improvement, but important outputs may still benefit from human review.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
